# market-abm 互動探索

快速跑一次模擬並視覺化結果。

In [ ]:
import sys
sys.path.insert(0, '..')

from data.fetch import fetch_ohlcv
from sim.simulation import run_simulation
from sim.metrics import compare
import matplotlib.pyplot as plt
import pandas as pd

# 下載資料
df = fetch_ohlcv('AAPL', '2022-01-01', '2025-01-01')
print(f'下載完成: {len(df)} 根')

In [ ]:
# 執行模擬
df_train = df.iloc[:-200].copy()
df_real_future = df.iloc[-200:].copy()

df_warmup, df_sim = run_simulation(
    df_real=df_train,
    sim_bars=200,
    warmup_bars=100,
    seed=42,
)
print('模擬完成:', df_sim.shape)

In [ ]:
# 統計比對
report = compare(df_real_future, df_sim)

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df_warmup['Close'].values, color='gray', label='warmup')
axes[0].plot(range(len(df_warmup), len(df_warmup)+len(df_sim)),
             df_sim['Close'].values, color='steelblue', label='sim')
axes[0].plot(range(len(df_warmup), len(df_warmup)+len(df_real_future)),
             df_real_future['Close'].values, color='gold', label='real', alpha=0.7)
axes[0].legend(); axes[0].set_title('Close Price')

import numpy as np
sim_rets  = np.diff(np.log(df_sim['Close'].values))
real_rets = np.diff(np.log(df_real_future['Close'].values))
axes[1].hist(real_rets, bins=50, alpha=0.5, density=True, color='gold', label='real')
axes[1].hist(sim_rets,  bins=50, alpha=0.5, density=True, color='steelblue', label='sim')
axes[1].legend(); axes[1].set_title('Return Distribution')
plt.tight_layout(); plt.show()